In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from datetime import datetime

In [10]:
df = pd.read_csv("/content/customer_sales.csv")

In [11]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (2240, 29)


In [12]:
print("\nFirst 5 rows:")
print(df.head())


First 5 rows:
     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
0    4/9/2012       58       635  ...                  7             0   
1    8/3/2014       38        11  ...                  5             0   
2  21-08-2013       26       426  ...                  4             0   
3   10/2/2014       26        11  ...                  6             0   
4  19-01-2014       94       173  ...                  5             0   

   AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Complain  \


In [13]:
print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Missing values per column:
Income    24
dtype: int64


In [14]:
df = df.dropna(subset=["Income"])
print("\nShape after dropping missing Income rows:", df.shape)


Shape after dropping missing Income rows: (2216, 29)


In [15]:
df = df.drop(columns=["Z_CostContact", "Z_Revenue", "ID"], errors="ignore")

In [17]:
df = df[df["Income"] < 200000]
print("Shape after removing income outliers:", df.shape)

Shape after removing income outliers: (2215, 26)


In [19]:
df = df[df["Year_Birth"] > 1940]
print("Shape after removing income outliers:", df.shape)


Shape after removing income outliers: (2211, 26)


In [24]:
df["Age"] = datetime.now().year - df["Year_Birth"]

In [26]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="mixed")
df["Days_As_Customer"] = (datetime.now() - df["Dt_Customer"]).dt.days

In [27]:
spending_cols = ["MntWines", "MntFruits", "MntMeatProducts",
                  "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df["Total_Spending"] = df[spending_cols].sum(axis=1)

In [28]:
purchase_cols = ["NumDealsPurchases", "NumWebPurchases",
                  "NumCatalogPurchases", "NumStorePurchases"]
df["Total_Purchases"] = df[purchase_cols].sum(axis=1)

In [29]:
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

In [30]:
df["Marital_Status"] = df["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner",
    "Single": "Single", "Divorced": "Single",
    "Widow": "Single", "Alone": "Single",
    "Absurd": "Single", "YOLO": "Single"
})

In [33]:
df = df.drop(columns=["Year_Birth", "Dt_Customer"], errors="ignore")
print("\nNew engineered columns added: Age, Days_As_Customer, Total_Spending, Total_Purchases, Total_Children")


New engineered columns added: Age, Days_As_Customer, Total_Spending, Total_Purchases, Total_Children


In [34]:
education_order = {
    "Basic": 0, "2n Cycle": 1, "Graduation": 2, "Master": 3, "PhD": 4
}
df["Education"] = df["Education"].map(education_order)

In [36]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Marital_Status"] = le.fit_transform(df["Marital_Status"])

In [37]:
cluster_data = df.select_dtypes(include=[np.number]).copy()
print("\nFinal columns going into clustering ({} columns):".format(cluster_data.shape[1]))
print(list(cluster_data.columns))


Final columns going into clustering (29 columns):
['Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response', 'Age', 'Days_As_Customer', 'Total_Spending', 'Total_Purchases', 'Total_Children']


In [38]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(cluster_data)

In [39]:
pca_full = PCA().fit(scaled_data)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
plt.axhline(y=0.95, color='r', linestyle='--', label="95% threshold")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("How many components explain 95% of the data?")
plt.legend()
plt.savefig("pca_variance.png")
plt.close()
print("\nSaved: pca_variance.png")


Saved: pca_variance.png


In [40]:
pca = PCA(n_components=3)
pca_data = pca.fit_transform(scaled_data)
print(f"Compressed from {scaled_data.shape[1]} columns down to {pca_data.shape[1]} components")
print("Variance explained by these 3 components:", round(sum(pca.explained_variance_ratio_) * 100, 1), "%")

Compressed from 29 columns down to 3 components
Variance explained by these 3 components: 45.6 %


In [41]:
# Elbow Method
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    km.fit(pca_data)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method")
plt.savefig("elbow_method.png")
plt.close()
print("Saved: elbow_method.png")

Saved: elbow_method.png


In [42]:
# Silhouette Score
silhouette_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init="k-means++", random_state=42, n_init=10)
    labels = km.fit_predict(pca_data)
    score = silhouette_score(pca_data, labels)
    silhouette_scores.append(score)
    print(f"K={k} -> Silhouette Score = {score:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), silhouette_scores, marker='o', color='green')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score per K")
plt.savefig("silhouette_scores.png")
plt.close()
print("Saved: silhouette_scores.png")

K=2 -> Silhouette Score = 0.474
K=3 -> Silhouette Score = 0.431
K=4 -> Silhouette Score = 0.436
K=5 -> Silhouette Score = 0.343
K=6 -> Silhouette Score = 0.319
K=7 -> Silhouette Score = 0.336
K=8 -> Silhouette Score = 0.330
K=9 -> Silhouette Score = 0.333
K=10 -> Silhouette Score = 0.317
Saved: silhouette_scores.png


In [46]:
BEST_K = 2

final_kmeans = KMeans(n_clusters=BEST_K, init="k-means++", random_state=42, n_init=10)
cluster_labels = final_kmeans.fit_predict(pca_data)
df["Cluster"] = cluster_labels

print("\nCustomers per cluster:")
print(df["Cluster"].value_counts())


Customers per cluster:
Cluster
1    1306
0     905
Name: count, dtype: int64


In [47]:
#VISUALIZE

plt.figure(figsize=(8, 6))
plt.scatter(pca_data[:, 0], pca_data[:, 1], c=cluster_labels, cmap="viridis")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title(f"Customer Clusters (K={BEST_K})")
plt.savefig("cluster_plot.png")
plt.close()
print("Saved: cluster_plot.png")

Saved: cluster_plot.png


In [52]:
#BUSINESS PERSONAS

persona_columns = ["Age", "Income", "Total_Spending", "Total_Purchases",
                   "Total_Children", "Education", "Days_As_Customer"]

cluster_summary = df.groupby("Cluster")[persona_columns].mean().round(1)

print("\nCLUSTER PERSONA SUMMARY")
print(cluster_summary)


CLUSTER PERSONA SUMMARY
          Age   Income  Total_Spending  Total_Purchases  Total_Children  \
Cluster                                                                   
0        59.1  71166.6          1230.2             21.8             0.5   
1        55.7  38649.3           175.9             10.1             1.3   

         Education  Days_As_Customer  
Cluster                               
0              2.5            4840.6  
1              2.4            4794.1  


In [54]:
# BUSINESS PERSONAS

print("\nBUSINESS PERSONAS")

print("""
Cluster 0: High-Value Customers
- Higher average income
- Much higher total spending
- More purchases
- Lower average number of children
Business Action:
- Offer premium products and loyalty rewards
- Provide personalized offers
- Encourage repeat purchases through VIP/loyalty programs



Cluster 1: Budget-Conscious Customers
- Lower average income
- Much lower total spending
- Fewer purchases
- Higher average number of children
Business Action:
- Offer discounts and budget-friendly products
- Use family-oriented promotions
- Provide bundle deals to encourage larger purchases
""")


BUSINESS PERSONAS

Cluster 0: High-Value Customers
- Higher average income
- Much higher total spending
- More purchases
- Lower average number of children
Business Action:
- Offer premium products and loyalty rewards
- Provide personalized offers
- Encourage repeat purchases through VIP/loyalty programs



Cluster 1: Budget-Conscious Customers
- Lower average income
- Much lower total spending
- Fewer purchases
- Higher average number of children
Business Action:
- Offer discounts and budget-friendly products
- Use family-oriented promotions
- Provide bundle deals to encourage larger purchases

